In [37]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import altair as alt

checklist_path = Path("../data/raw/ebird/cook_ebird_checklists_2020_2025.txt")

print("Dataset found:", checklist_path.exists())

Dataset found: True


In [17]:
#loading all Cook County checklists
checklists = pd.read_csv(checklist_path, sep="\t", low_memory=False)

# Convert observation dates to datetime
checklists["OBSERVATION DATE"] = pd.to_datetime(checklists["OBSERVATION DATE"])

print("Rows and columns:", checklists.shape)

Rows and columns: (418368, 34)


In [18]:
for i, col in enumerate(checklists.columns, start=1):
    print(f"{i}. {col}")

1. LAST EDITED DATE
2. COUNTRY
3. COUNTRY CODE
4. STATE
5. STATE CODE
6. COUNTY
7. COUNTY CODE
8. IBA CODE
9. BCR CODE
10. USFWS CODE
11. ATLAS BLOCK
12. LOCALITY
13. LOCALITY ID
14. LOCALITY TYPE
15. LATITUDE
16. LONGITUDE
17. OBSERVATION DATE
18. TIME OBSERVATIONS STARTED
19. OBSERVER ID
20. OBSERVER ORCID ID
21. SAMPLING EVENT IDENTIFIER
22. OBSERVATION TYPE
23. PROTOCOL NAME
24. PROTOCOL CODE
25. PROJECT NAMES
26. PROJECT IDENTIFIERS
27. DURATION MINUTES
28. EFFORT DISTANCE KM
29. EFFORT AREA HA
30. NUMBER OBSERVERS
31. ALL SPECIES REPORTED
32. GROUP IDENTIFIER
33. CHECKLIST COMMENTS
34. Unnamed: 33


In [19]:
columns_to_explore = [
    "LOCALITY TYPE",
    "OBSERVATION TYPE",
    "ALL SPECIES REPORTED"
]

for column in columns_to_explore:
    print(f"\n{column}")
    display(checklists[column].value_counts(dropna=False))
    
    # locality type = (personal locations, P), choose existing hotspot locations from a map (H), 
    # or choose to submit data for a town (T), postal code (PC), county (C), or state (S).


LOCALITY TYPE


LOCALITY TYPE
H     239859
P     177536
T        772
PC       201
Name: count, dtype: int64


OBSERVATION TYPE


OBSERVATION TYPE
Traveling                      241197
Stationary                     107951
Incidental                      63471
Area                             3252
Historical                       1982
eBird Pelagic Protocol            452
Nocturnal Flight Call Count        32
Banding                            31
Name: count, dtype: int64


ALL SPECIES REPORTED


ALL SPECIES REPORTED
1    339970
0     78398
Name: count, dtype: int64

In [20]:
pd.crosstab(
    checklists["OBSERVATION TYPE"],
    checklists["ALL SPECIES REPORTED"],
    normalize="index"
).round(3)

ALL SPECIES REPORTED,0,1
OBSERVATION TYPE,,
Area,0.004,0.996
Banding,0.032,0.968
Historical,0.362,0.638
Incidental,0.995,0.005
Nocturnal Flight Call Count,0.406,0.594
Stationary,0.046,0.954
Traveling,0.040,0.960
eBird Pelagic Protocol,0.007,0.993


In [38]:
print("Earliest date:", checklists["OBSERVATION DATE"].min())
print("Latest date:", checklists["OBSERVATION DATE"].max())

Earliest date: 2020-01-01 00:00:00
Latest date: 2025-01-31 00:00:00


In [ ]:
# this is just checklist submissions, not unique birders or birding outings

january = checklists[
    checklists["OBSERVATION DATE"].dt.month == 1
].copy()

january_counts = (
    january.groupby(january["OBSERVATION DATE"].dt.year)
    .size()
    .reset_index(name="checklists")
)

display(january_counts)

,OBSERVATION DATE,checklists
0,2020,3305
1,2021,5014
2,2022,5525
3,2023,5360
4,2024,6981
5,2025,6111


In [23]:

#Remove January
checklists["OBSERVATION DATE"] = pd.to_datetime(
    checklists["OBSERVATION DATE"]
)

checklists_2020_2024 = checklists[
    checklists["OBSERVATION DATE"].dt.year.between(2020, 2024)
].copy()

print("Total checklists:", len(checklists_2020_2024))

display(
    pd.crosstab(
        checklists_2020_2024["OBSERVATION TYPE"],
        checklists_2020_2024["ALL SPECIES REPORTED"],
        normalize="index"
    ).round(3)
)

Total checklists: 412257


ALL SPECIES REPORTED,0,1
OBSERVATION TYPE,,
Area,0.004,0.996
Banding,0.032,0.968
Historical,0.368,0.632
Incidental,0.995,0.005
Nocturnal Flight Call Count,0.406,0.594
Stationary,0.046,0.954
Traveling,0.040,0.960
eBird Pelagic Protocol,0.007,0.993


In [24]:
checklists_2020_2024["OBSERVATION DATE"].dt.year.value_counts().sort_index()

OBSERVATION DATE
2020     62212
2021     76941
2022     81338
2023     87412
2024    104354
Name: count, dtype: int64

In [ ]:
community_areas = gpd.read_file(
    "../data/processed/community_areas.geojson"
)

print("Number of geographic features:", len(community_areas))
print("Coordinate system:", community_areas.crs)
print("Columns:", community_areas.columns.tolist())

Number of geographic features: 77
Coordinate system: EPSG:4326
Columns: ['community', 'shape_area', 'area_num_1', 'area_numbe', 'shape_len', 'community_area', 'geometry']


In [46]:
# convert checklist coordinates into geographic points
checklist_points = gpd.GeoDataFrame(
    checklists_2020_2024,
    geometry=gpd.points_from_xy(
        checklists_2020_2024["LONGITUDE"],
        checklists_2020_2024["LATITUDE"]
    ),
    crs="EPSG:4326"
)

# Match each checklist to its Chicago community area
chicago_checklists = gpd.sjoin(
    checklist_points,
    community_areas[["community", "geometry"]],
    how="inner",
    predicate="within"
)


print("Cook County checklists:", len(checklists_2020_2024))
print("Chi checklists:", len(chicago_checklists))
print("Comm areas represented:", chicago_checklists["community"].nunique())

Cook County checklists: 412257
Chi checklists: 218781
Comm areas represented: 77


In [ ]:
chicago_yearly = (
    chicago_checklists
    .groupby(chicago_checklists["OBSERVATION DATE"].dt.year)
    .size()
    .reset_index(name="checklists")
)

display(chicago_yearly)



,OBSERVATION DATE,checklists
0,2020,27159
1,2021,39800
2,2022,42781
3,2023,45962
4,2024,63079


(218781, 37)

SAVING DATASET 

In [34]:
# Save Chicago checklist data, including community-area assignments
chicago_checklists.drop(
    columns=["geometry", "index_right"],
    errors="ignore"
).to_csv(
    "../data/processed/chicago_checklists_2020_2024.csv",
    index=False
)

# Save annual checklist totals
chicago_yearly.to_csv(
    "../data/processed/chicago_yearly_2020_2024.csv",
    index=False
)

print("Chicago datasets saved.")


Chicago datasets saved.


What am I comparing?
What type of data is on each axis or encoding channel?
Why is this graphical form better than a simpler alternative?
What might a reader incorrectly infer?
What does this visual contribute to the larger story?

Checklist count	How much reporting activity occurred?
Distinct observer IDs	How many identifiable contributors submitted records?
Birding duration	How much observation time was reported?
Complete checklist count	How much potentially more informative survey coverage occurred?
Reported species richness	How many distinct species were documented?

TROUBLE SHOOTING VARIABLES
sampling event identifier: one submitted checklist, normally unique in data
observer id: one ebird account, one account can submit many checklists
- each count number of rows in each comm-area group
# are a relatively small amount of bird watchers responsible for a large amount of checklists?

In [59]:

# how much birding activity is being reported relative to how many distinct ewbird accounts are contributing to it 
# briding report submissions is the volume of reporting
# birder is the number of different accounts affiliated w ebird 

community_counts = (
    chicago_checklists.groupby("community")
    .agg(
        birder=("OBSERVER ID", "nunique"), #every checklist row even if no observer id
        birding_report_submissions=("SAMPLING EVENT IDENTIFIER", "nunique")#unique ids
    )
    .reset_index()
)

display(community_counts)

,community,birder,birding_report_submissions
0,ALBANY PARK,369,5721
1,ARCHER HEIGHTS,30,328
2,ARMOUR SQUARE,118,253
3,ASHBURN,21,255
4,AUBURN GRESHAM,10,20
...,...,...,...
72,WEST LAWN,23,90
73,WEST PULLMAN,117,401
74,WEST RIDGE,386,4218
75,WEST TOWN,956,7093


In [ ]:
community_chart = alt.Chart(community_counts).mark_bar().encode(
    x=alt.X("birding_report_submissions:Q", title="Checklist submissions"),
    y=alt.Y("community:N", sort="-x", title="Community area")
).properties(
    title="eBird Checklist Submissions by Chicago Community Area, 2020–2024",
    width=600,
    height=1200
)

community_chart

# This chart has a scale issue.

alt.Chart(...)

In [62]:
display(
    community_counts[
        community_counts["community"].isin(["AVALON PARK", "BURNSIDE"])
    ]
)

,community,birder,birding_report_submissions
6,AVALON PARK,14,15
12,BURNSIDE,1,1


In [52]:
# are a relatively small amount of bird watchers responsible for a large amount of checklists?

area_counts = (
    chicago_checklists.groupby("community")
    .agg(
        checklists=("OBSERVER ID", "size"), #every checklist row even if no observer id
        observers=("OBSERVER ID", "nunique")#unique ids
    )
    .reset_index()
)

display(area_counts)

,community,checklists,observers
0,ALBANY PARK,5721,369
1,ARCHER HEIGHTS,328,30
2,ARMOUR SQUARE,253,118
3,ASHBURN,255,21
4,AUBURN GRESHAM,20,10
...,...,...,...
72,WEST LAWN,90,23
73,WEST PULLMAN,401,117
74,WEST RIDGE,4218,386
75,WEST TOWN,7093,956
